In [1]:
import os
import cv2
import torch

import numpy as np
import pandas as pd 

from matplotlib import pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split

%pip install efficientnet-pytorch
from efficientnet_pytorch import EfficientNet

  Created wheel for efficientnet-pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16446 sha256=a36a48f6da9eec9b75a38b3129c2d8dedaf422816dd45a678450886c8beeb04b
  Stored in directory: /root/.cache/pip/wheels/63/17/7e/07f1c55a623c96ba9b291c5d2e4901afffb953b032c2470ca6
Successfully built efficientnet-pytorch
Note: you may need to restart the kernel to use updated packages.


In [23]:
NAME = 'efficientnet-b4'
BATCH_SIZE = 128
RESUME=False

import gc;gc.collect()
BASE_LR=0.0001
VAL_SIZE = 0.1

In [24]:
class CelebDataset(torch.utils.data.Dataset):
    def __init__(self, metadata):
        self.metadata = metadata

    def __getitem__(self, ix):
        sample = self.metadata.iloc[ix]

        face = cv2.imread(os.path.join('/kaggle/input/celeba-dataset//img_align_celeba/img_align_celeba', sample.image_id))
        #face = cv2.resize(face, (int(face.shape[1]/ 2), int(face.shape[0]/ 2)))
        face = face[45:173, 25:153] # center_crop
        face = cv2.cvtColor(face, cv2.COLOR_BGR2RGB)
        face = np.moveaxis(face, -1, 0)

        face = (face.astype(np.float32) - 128) / 128
        face = face*(1-np.random.rand()/ 100.) # random brightness
    
        
        labels = sample.drop('image_id').values

        return face, labels.astype(np.float)

    def __len__(self):
        return len(self.metadata)

In [25]:
keep_cols = [
    'image_id',
    'Male',
    'Bald', 'Bangs', 'Black_Hair', 'Blond_Hair', 'Brown_Hair', 'Gray_Hair',
    'Straight_Hair', 'Wavy_Hair', 'Receding_Hairline',
    'Mustache', 'Sideburns', 'No_Beard',
    'Eyeglasses', 'Wearing_Hat', 'Wearing_Earrings',
    'Wearing_Necklace', 'Wearing_Necktie'
]

In [26]:
data = pd.read_csv('/kaggle/input/celeba-dataset/list_attr_celeba.csv').replace({-1: 0})
data = data[keep_cols]
train, val = train_test_split(data, test_size=VAL_SIZE, random_state=420)

class_distribution = train.describe().transpose()[['mean']].rename({'mean': 'train_mean'}, axis=1).join(
    val.describe().transpose()[['mean']].rename({'mean': 'val_mean'}, axis=1)
)
class_distribution['diff'] = class_distribution['train_mean'] - class_distribution['val_mean']
class_distribution.sort_values('train_mean')

,train_mean,val_mean,diff
Bald,0.022447,0.022409,0.000039
Mustache,0.041472,0.042201,-0.000729
Gray_Hair,0.041889,0.042498,-0.000609
Wearing_Hat,0.048465,0.048421,0.000044
Sideburns,0.056406,0.057453,-0.001047
Eyeglasses,0.065159,0.064758,0.000401
Wearing_Necktie,0.072853,0.071471,0.001382
Receding_Hairline,0.079643,0.080997,-0.001354
Wearing_Necklace,0.123128,0.121520,0.001608
Blond_Hair,0.148213,0.146002,0.002211


In [27]:
val_dl =  torch.utils.data.DataLoader(
    CelebDataset(val.reset_index(drop=True)),
    num_workers=torch.multiprocessing.cpu_count() - 1,
    pin_memory=True,
    batch_size=BATCH_SIZE
)

train_dl =  torch.utils.data.DataLoader(
    CelebDataset(train.reset_index(drop=True)),
    num_workers=torch.multiprocessing.cpu_count() - 1,
    shuffle=True,
    pin_memory=True,
    batch_size=BATCH_SIZE
)

In [28]:
import torch

BATCH_NORM_MODULES = {
    torch.nn.BatchNorm1d,
    torch.nn.BatchNorm2d,
    torch.nn.BatchNorm3d
}
def freeze_module(mod):
    for param in mod.parameters():
        param.requires_grad = False

def unfreeze_module(mod):
    for param in mod.parameters():
        param.requires_grad = True

def unfreeze_batch_norm(model):
    for name, mod in model.named_modules():
        if type(mod) in BATCH_NORM_MODULES:
            unfreeze_module(mod)

def unfreeze_linear(model):
    for name, mod in model.named_modules():
        if type(mod) == torch.nn.modules.linear.Linear:
            unfreeze_module(mod)


In [29]:
len(data.columns)

19

In [30]:
from efficientnet_pytorch import EfficientNet

model = EfficientNet.from_pretrained(NAME, num_classes=len(data.columns) - 1)
model._fc = torch.nn.Linear(in_features=model._fc.in_features, out_features=18, bias=False)


if RESUME:
    model.load_state_dict(torch.load('/kaggle/input/dfdcmodels/efficientnet-b4_mymodel_7125.pth'))
    
model._fc = torch.nn.Linear(in_features=model._fc.in_features, out_features=len(data.columns) - 1, bias=False)
model.train()
model.cuda()

if not RESUME:
    freeze_module(model)
    unfreeze_batch_norm(model)
    unfreeze_linear(model)

Loaded pretrained weights for efficientnet-b4


In [31]:
from ignite.engine import Events, create_supervised_trainer, create_supervised_evaluator
from ignite.handlers import ModelCheckpoint
from ignite.metrics import Accuracy, Loss
from ignite.contrib.handlers.tqdm_logger import ProgressBar
from ignite.contrib.metrics import ROC_AUC

optimizer = torch.optim.Adam(model.parameters(), lr=BASE_LR)
loss = torch.nn.BCEWithLogitsLoss(wei ght=torch.Tensor(1 / data.drop('image_id', axis=1).values.mean(axis=0)).cuda())

trainer = create_supervised_trainer(
    model, optimizer, loss, device='cuda'
)

def thresholded_output_transform(output):
    y_pred, y = output
    y_pred = (y_pred > 0).to(np.float)
    return y_pred, y

evaluator = create_supervised_evaluator(
    model, metrics={'loss': Loss(loss), 'accuracy': Accuracy(thresholded_output_transform), 'roc_auc': ROC_AUC()}, device='cuda'
)

model_checkpoint = ModelCheckpoint('tmp/models', NAME, create_dir=True, n_saved=15, require_empty=False)
trainer.add_event_handler(Events.ITERATION_COMPLETED(every=1425), model_checkpoint, {'mymodel': model})

pbar = ProgressBar()
pbar.attach(trainer)

@trainer.on(Events.ITERATION_COMPLETED)
def log_training_loss(trainer):
    pbar.pbar.set_postfix({
        'batch': tuple(trainer.state.batch[0].shape)[:-2],
        'loss': trainer.state.output
    })
    if trainer.state.iteration == 1000:
        unfreeze_module(model)
    
history = []
@trainer.on(Events.ITERATION_COMPLETED(every=1425))
def log_validation_results(trainer):
    model.eval()
    evaluator.run(val_dl)
    metrics = evaluator.state.metrics
    metrics = {
        'epoch':       trainer.state.epoch,
        'step':        trainer.state.iteration,
        'accuracy':    metrics['accuracy'],
        'val_loss':    metrics['loss'],
        'val_roc_auc': metrics['roc_auc'],
        'train_loss':  trainer.state.output,
    }
    print(metrics)
    history.append(metrics)
    
    model.train()

    
trainer.run(train_dl, max_epochs=10)

{'epoch': 1, 'step': 1425, 'accuracy': 0.9190057036305802, 'val_loss': 1.4237462965771213, 'val_roc_auc': 0.9278713939607687, 'train_loss': 1.4754231350708007}


{'epoch': 2, 'step': 2850, 'accuracy': 0.9351431391905232, 'val_loss': 1.155748741950292, 'val_roc_auc': 0.9524763797970294, 'train_loss': 1.3453373132376805}


{'epoch': 3, 'step': 4275, 'accuracy': 0.9379044641877811, 'val_loss': 1.1016869243207776, 'val_roc_auc': 0.9561332128113688, 'train_loss': 0.8375242743962127}


{'epoch': 4, 'step': 5700, 'accuracy': 0.9378002632444883, 'val_loss': 1.1034358982099266, 'val_roc_auc': 0.9572853766511584, 'train_loss': 0.9089085246792645}


{'epoch': 5, 'step': 7125, 'accuracy': 0.9381622244159263, 'val_loss': 1.1225134754899684, 'val_roc_auc': 0.9570110067346028, 'train_loss': 0.8963379352936958}


...

{'epoch': 7, 'step': 9975, 'accuracy': 0.9369858506087528, 'val_loss': 1.2645761308053376, 'val_roc_auc': 0.9549462903710833, 'train_loss': 0.7053458298795741}


{'epoch': 8, 'step': 11400, 'accuracy': 0.936190632883624, 'val_loss': 1.3475607853661078, 'val_roc_auc': 0.9546381625826241, 'train_loss': 0.581413662284995}


{'epoch': 9, 'step': 12825, 'accuracy': 0.9366129209169684, 'val_loss': 1.4432256338506748, 'val_roc_auc': 0.9526371189880304, 'train_loss': 0.7057876989104144}


{'epoch': 10, 'step': 14250, 'accuracy': 0.9351952396621696, 'val_loss': 1.5203516357491442, 'val_roc_auc': 0.9515377927256962, 'train_loss': 0.45589291739976767}


State:
	iteration: 14250
	epoch: 10
	epoch_length: 1425
	max_epochs: 10
	output: 0.45589291739976767
	batch: <class 'list'>
	metrics: <class 'dict'>
	dataloader: <class 'torch.utils.data.dataloader.DataLoader'>
	seed: 12

In [33]:
pd.DataFrame(history).to_csv('history.csv', index=False)

In [34]:
pd.DataFrame(history)

,epoch,step,accuracy,val_loss,val_roc_auc,train_loss
0,1,1425,0.919006,1.423746,0.927871,1.475423
1,2,2850,0.935143,1.155749,0.952476,1.345337
2,3,4275,0.937904,1.101687,0.956133,0.837524
3,4,5700,0.937800,1.103436,0.957285,0.908909
4,5,7125,0.938162,1.122513,0.957011,0.896338
5,6,8550,0.937680,1.197858,0.956108,0.742500
6,7,9975,0.936986,1.264576,0.954946,0.705346
7,8,11400,0.936191,1.347561,0.954638,0.581414
8,9,12825,0.936613,1.443226,0.952637,0.705788
9,10,14250,0.935195,1.520352,0.951538,0.455893
